<a href="https://colab.research.google.com/github/kanish-27/KanishKrishna-Codeboosters-Internship-2026/blob/main/Phase_01_Data_Engineering/Day_04_BigData_PySpark_Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

In [3]:
df=pd.read_csv("/content/Housing.csv")

In [4]:
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [7]:
x=df[['bedrooms','bathrooms','stories']]

In [6]:
y=df['price']

In [8]:
df.isnull().any()

,0
price,False
area,False
bedrooms,False
bathrooms,False
stories,False
mainroad,False
guestroom,False
basement,False
hotwaterheating,False
airconditioning,False


In [9]:
from sklearn.model_selection import train_test_split

In [10]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [11]:
from sklearn.linear_model import LinearRegression

In [22]:
model=LinearRegression()

In [13]:
model.fit(x_train,y_train)

LinearRegression()

In [14]:
y_predict=model.predict(x_test)

In [15]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [16]:
mean_absolute_error(y_test,y_predict)

1088079.9115446832

In [17]:
mean_squared_error(y_test,y_predict)

1886087117574.2532

In [18]:
r2_score(y_test,y_predict)

0.12026073883538124

In [19]:
import joblib

In [20]:
joblib.dump(model,"linear.pkl")

['linear.pkl']

In [21]:
!pip install pyspark --quiet

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
spark=SparkSession.builder \
.appName('Day4_BigData_Sales') \
.config('spark.sql.adaptive.enabled','true') \
.getOrCreate()
print(f'Spark version : {spark.version}')
print(f'SparkSesion : ACTIVE')
print(f'Application : {spark.sparkContext.appName}')

Spark version : 4.0.2
SparkSesion : ACTIVE
Application : Day4_BigData_Sales


In [ ]:
cd_bronze=spark.read \
.option('header','true') \
.option('inferSchema','true') \
.csv('/content/large_sales_data.csv')
print('=== BRONZE LAYER - RAW Data ===')
print(f'Rows : {cd_bronze.count()}')
print(f'Columns :{len(cd_bronze.columns)}')
print(f'Names : {cd_bronze.columns}')
print()
cd_bronze.printSchema()


=== BRONZE LAYER - RAW Data ===
Rows : 5000
Columns :13
Names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
print("First 5 rows:")
cd_bronze.show(5,truncate=False)
print('\nBasic statistics for numeric columns:')
cd_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [ ]:
cd_bronze.write \
.mode('overwrite') \
.parquet('sales_bronze.parquet')